In [13]:
import glob
import os
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

plt.style.use("ggplot")
plt.figure(figsize=(15, 10))

<Figure size 1500x1000 with 0 Axes>

<Figure size 1500x1000 with 0 Axes>

In [3]:
# checked for missing rows


all_files = glob.glob("raw_data/CRMLSSold*.csv")
all_dfs = []

for file in all_files:
    df_file = pd.read_csv(file, low_memory=False)``
    all_dfs.append(df_file)

raw_master_df = pd.concat(all_dfs, ignore_index=True)

is_sfr = (raw_master_df["PropertyType"] == "Residential") & (raw_master_df["PropertySubType"] == "SingleFamilyResidence")
sfr_df = raw_master_df[is_sfr]

features = ["ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeArea"]
sfr_features_df = sfr_df[features]

print("--- Missing Value Counts ---")
sfr_features_df.isnull().sum()

--- Missing Value Counts ---


ClosePrice                  2
LivingArea                147
BedroomsTotal               0
BathroomsTotalInteger      38
LotSizeArea              4749
dtype: int64

In [6]:
#checked which cities had the most missing lot size area since it was the most important

missing_lot_mask = sfr_df["LotSizeArea"].isnull()
missing_lot_df = sfr_df[missing_lot_mask]


print("--- Top cities w/ missing lot sizes ---")

print(missing_lot_df["City"].value_counts().head(10))

print("\n")
print("--- ClosePrice summary ---")
print(missing_lot_df["ClosePrice"].describe())

--- Top cities w/ missing lot sizes ---
City
San Diego        1754
Escondido         248
Chula Vista       247
El Cajon          188
Oceanside         167
Poway             129
La Mesa           127
Spring Valley     119
La Jolla          113
Carlsbad          107
Name: count, dtype: int64


--- ClosePrice Summary for Missing Lot Rows ---
count    4.749000e+03
mean     1.348780e+06
std      1.423160e+06
min      9.600000e+04
25%      7.900000e+05
50%      9.990000e+05
75%      1.446990e+06
max      5.000000e+07
Name: ClosePrice, dtype: float64


In [10]:
# imputed and created a flag collumn 

cleaned_df = sfr_df.copy()
cleaned_df["Missing_LotSizeArea"] = cleaned_df["LotSizeArea"].isnull().astype(int)

global_lotSize_median = cleaned_df["LotSizeArea"].median()

city_medians = cleaned_df.groupby("City")["LotSizeArea"].median()
# print(city_medians)

def fill_with_city_median(row):
    if pd.isnull(row["LotSizeArea"]):
        city = row["City"]
        return city_medians.get(city, global_lotSize_median)
    else:
        return row["LotSizeArea"]

cleaned_df["LotSizeArea"] = cleaned_df.apply(fill_with_city_median, axis=1)

In [19]:
# one hot encoded the only categorical variable which is the city 

ohe = OneHotEncoder(drop='first', sparse_output=False)

city_matrix = ohe.fit_transform(cleaned_df[['City']])

city_feature_names = ohe.get_feature_names_out(['City'])

city_encoded_df = pd.DataFrame(
    city_matrix, 
    columns = city_feature_names, 
    index = cleaned_df.index,
    dtype = int
)

final_features = [
    "ClosePrice", 
    "LivingArea", 
    "BedroomsTotal", 
    "BathroomsTotalInteger", 
    "LotSizeArea", 
    "Missing_LotSizeArea"
]

processed_df = pd.concat([cleaned_df[final_features], city_encoded_df], axis=1)

print(f"Dataset shape: {processed_df.shape}")
# processed_df.head()
# processed_df[processed_df['City_Acampo'] == 1].head(5)

Dataset shape: (278241, 1112)


,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeArea,Missing_LotSizeArea,City_Acampo,City_Acton,City_Adelanto,City_Adin,City_Agoura,City_Agoura Hills,City_Agua Dulce,City_Agua Fria,City_Aguanga,City_Ahwahnee,City_Alameda,City_Alamo,City_Albany,City_Albion,City_Alhambra,City_Aliso Viejo,City_Almanor,City_Alpine,City_Alta Loma,City_Altadena,City_Alturas,City_Alviso,City_Amador City,City_American Canyon,City_Anaheim,City_Anaheim Hills,City_Anderson,City_Angels Camp,City_Angelus Oaks,City_Angwin,City_Antelope,City_Antioch,City_Anza,City_Apple Valley,City_Aptos,City_Arbuckle,City_Arcadia,City_Arcata,City_Arleta,City_Arnold,City_Aromas,City_Arrowbear,City_Arroyo Grande,City_Artesia,...,City_Weldon,City_West Covina,City_West Hills,City_West Hollywood,City_West Los Angeles,City_West Point,City_West Sacramento,City_West Toluca Lake,City_Westchester,City_Westlake Village,City_Westminster,City_Westport,City_Westwood,City_Westwood - Century City,City_Wheatland,City_Whitewater,City_Whitmore,City_Whittier,City_Wildomar,City_Williams,City_Willits,City_Willows,City_Wilmington,City_Wilton,City_Winchester,City_Windsor,City_Windsor Hills,City_Winnetka,City_Winterhaven,City_Winters,City_Winton,City_Wishon,City_Wofford Heights,City_Woodbridge,City_Woodcrest,City_Woodlake,City_Woodland,City_Woodland Hills,City_Woodside,City_Wrightwood,City_Yermo,City_Yorba Linda,City_Yorkville,City_Yosemite,City_Yountville,City_Yreka,City_Yuba City,City_Yucaipa,City_Yucca Valley,City_nan
72680,850000.0,2041.0,3.0,2.0,226947.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
447704,998000.0,1955.0,3.0,3.0,219542.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
547251,840000.0,2652.0,3.0,3.0,211266.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
